In [ ]:
# %pip install --user imbalanced-learn

In [2]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

In [ ]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON RandomForestClassifier

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_20_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()


model_oh = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)
model_kmers = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5697
ROC-AUC k-mers: 0.6109
PR-AUC One-Hot: 0.2623
PR-AUC k-mers: 0.2882
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.61      0.78      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.61      0.78      0.68       770



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/si

In [2]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_20_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()

scaler = StandardScaler()
X_train_kmers = scaler.fit_transform(X_train_kmers)
X_test_kmers = scaler.transform(X_test_kmers)



model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


ROC-AUC One-Hot: 0.5280
ROC-AUC k-mers: 0.5923
PR-AUC One-Hot: 0.2323
PR-AUC k-mers: 0.2963
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      0.86      0.82       599
           1       0.23      0.15      0.18       171

    accuracy                           0.70       770
   macro avg       0.51      0.50      0.50       770
weighted avg       0.66      0.70      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.79      0.87      0.83       599
           1       0.31      0.19      0.24       171

    accuracy                           0.72       770
   macro avg       0.55      0.53      0.53       770
weighted avg       0.68      0.72      0.70       770



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
# %pip install --user tensorflow

In [3]:
# PRUEBA CON LSTM BIDIRECCIONAL

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.metrics import AUC

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_50_OR_3109_con_rsid_extendida.csv")

etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)

X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

dicc = {"a": 1, "c": 2, "g": 3, "t": 4}

def codificador(secuencia):
    return np.array([dicc.get(nuc, 0) for nuc in secuencia])

X_cod = np.array([codificador(secuencia) for secuencia in X_seq])

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 2026)
id_train_val, id_test = next(gss1.split(X_cod, y, groups = grupos_rsid))

X_tv = X_cod[id_train_val]
y_tv = y[id_train_val]
grupos_tv = grupos_rsid[id_train_val]

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
id_train, id_val = next(gss2.split(X_tv, y_tv, groups = grupos_tv))

X_train, y_train = X_tv[id_train], y_tv[id_train]
X_val, y_val = X_tv[id_val], y_tv[id_val]
X_test, y_test = X_cod[id_test], y[id_test]

pesos_clases = compute_class_weight(class_weight = "balanced", classes = np.unique(y_train), y = y_train)

dicc_pesos = {0: pesos_clases[0], 1: pesos_clases[1]}

longitud_sec = len(base.iloc[0]["Secuencia"])
tam_vocabulario = 5
dim_embedding = 16

model = Sequential([

    Embedding(input_dim = tam_vocabulario,
              output_dim = dim_embedding,
              input_length = longitud_sec),

    Bidirectional(LSTM(units = 32, dropout = 0.3, recurrent_dropout = 0.3)),

    Dropout(0.3),

    Dense(units = 1, activation = "sigmoid")
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss = "binary_crossentropy",
    metrics = ["accuracy", AUC(name = "auc_roc"), AUC(name = "auc_pr", curve = "PR")]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = "val_auc_pr",
    mode = "max",
    patience = 10,
    restore_best_weights = True
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = 50, 
    batch_size = 64, 
    class_weight = dicc_pesos,
    callbacks = [early_stop],
    verbose = 1
)

print("RESULTADOS FINALES")

y_pred_proba = model.predict(X_test).flatten()
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

print(f"ROC-AUC Test: {roc_auc_score(y_test, y_pred_proba): .4f}")
print(f"PR-AUC Test: {average_precision_score(y_test, y_pred_proba): .4f}\n")

print(classification_report(y_test, y_pred_bin))

2026-06-04 20:08:17.685041: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-04 20:08:17.723075: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-04 20:08:18.293623: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-06-04 20:08:18.796941: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-06-04 20:08:18.796966: I tensorflow/compiler/xla/stream_executor/cuda/

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 101, 16)           80        
                                                                 
 bidirectional (Bidirection  (None, 64)                12544     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 12689 (49.57 KB)
Trainable params: 12689 (49.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/50
30/30 [==============================] - 5s 78ms/step - loss: 0.6942 - accu

In [4]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON RandomForestClassifier y BASE AUMENTADA

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()


model_oh = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)
model_kmers = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5377
ROC-AUC k-mers: 0.5604
PR-AUC One-Hot: 0.5173
PR-AUC k-mers: 0.5394
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.53      0.54      0.54       751
           1       0.51      0.50      0.51       725

    accuracy                           0.52      1476
   macro avg       0.52      0.52      0.52      1476
weighted avg       0.52      0.52      0.52      1476

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.51      0.99      0.67       751
           1       0.62      0.02      0.04       725

    accuracy                           0.51      1476
   macro avg       0.57      0.50      0.36      1476
weighted avg       0.57      0.51      0.36      1476



In [5]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier Y BASE AUMENTADA

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()

scaler = StandardScaler()
X_train_kmers = scaler.fit_transform(X_train_kmers)
X_test_kmers = scaler.transform(X_test_kmers)



model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


ROC-AUC One-Hot: 0.5098
ROC-AUC k-mers: 0.5133
PR-AUC One-Hot: 0.4960
PR-AUC k-mers: 0.5003
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.51      0.52      0.52       751
           1       0.49      0.48      0.49       725

    accuracy                           0.50      1476
   macro avg       0.50      0.50      0.50      1476
weighted avg       0.50      0.50      0.50      1476

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.52      0.69      0.59       751
           1       0.50      0.33      0.40       725

    accuracy                           0.51      1476
   macro avg       0.51      0.51      0.49      1476
weighted avg       0.51      0.51      0.50      1476



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:
#PRUEBA LSTM BIDIRECCIONAL CON BASE AUMENTADA

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.metrics import AUC

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv")

etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)

X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

dicc = {"a": 1, "c": 2, "g": 3, "t": 4}

def codificador(secuencia):
    return np.array([dicc.get(nuc, 0) for nuc in secuencia])

X_cod = np.array([codificador(secuencia) for secuencia in X_seq])

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 2026)
id_train_val, id_test = next(gss1.split(X_cod, y, groups = grupos_rsid))

X_tv = X_cod[id_train_val]
y_tv = y[id_train_val]
grupos_tv = grupos_rsid[id_train_val]

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
id_train, id_val = next(gss2.split(X_tv, y_tv, groups = grupos_tv))

X_train, y_train = X_tv[id_train], y_tv[id_train]
X_val, y_val = X_tv[id_val], y_tv[id_val]
X_test, y_test = X_cod[id_test], y[id_test]

pesos_clases = compute_class_weight(class_weight = "balanced", classes = np.unique(y_train), y = y_train)

dicc_pesos = {0: pesos_clases[0], 1: pesos_clases[1]}

longitud_sec = len(base.iloc[0]["Secuencia"])
tam_vocabulario = 5
dim_embedding = 16

model = Sequential([

    Embedding(input_dim = tam_vocabulario,
              output_dim = dim_embedding,
              input_length = longitud_sec),

    Bidirectional(LSTM(units = 32, dropout = 0.3, recurrent_dropout = 0.3)),

    Dropout(0.3),

    Dense(units = 1, activation = "sigmoid")
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss = "binary_crossentropy",
    metrics = ["accuracy", AUC(name = "auc_roc"), AUC(name = "auc_pr", curve = "PR")]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = "val_auc_pr",
    mode = "max",
    patience = 10,
    restore_best_weights = True
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = 50, 
    batch_size = 64, 
    class_weight = dicc_pesos,
    callbacks = [early_stop],
    verbose = 1
)

print("RESULTADOS FINALES")

y_pred_proba = model.predict(X_test).flatten()
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

print(f"ROC-AUC Test: {roc_auc_score(y_test, y_pred_proba): .4f}")
print(f"PR-AUC Test: {average_precision_score(y_test, y_pred_proba): .4f}\n")

print(classification_report(y_test, y_pred_bin))

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 101, 16)           80        
                                                                 
 bidirectional_1 (Bidirecti  (None, 64)                12544     
 onal)                                                           
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 65        
                                                                 
Total params: 12689 (49.57 KB)
Trainable params: 12689 (49.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/50
59/59 [==============================] - 8s 76ms/step - loss: 0.6934 - ac

In [1]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON RandomForestClassifier y BASE REDUCIDA

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_50_OR_2400_reducida_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()


model_oh = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)
model_kmers = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5922
ROC-AUC k-mers: 0.6660
PR-AUC One-Hot: 0.3352
PR-AUC k-mers: 0.4145
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.71      1.00      0.83       428
           1       0.00      0.00      0.00       171

    accuracy                           0.71       599
   macro avg       0.36      0.50      0.42       599
weighted avg       0.51      0.71      0.60       599

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.72      0.97      0.83       428
           1       0.39      0.04      0.07       171

    accuracy                           0.71       599
   macro avg       0.55      0.51      0.45       599
weighted avg       0.62      0.71      0.61       599



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [2]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier Y BASE REDUCIDA

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_50_OR_2400_reducida_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()

scaler = StandardScaler()
X_train_kmers = scaler.fit_transform(X_train_kmers)
X_test_kmers = scaler.transform(X_test_kmers)



model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5532
ROC-AUC k-mers: 0.5889
PR-AUC One-Hot: 0.3089
PR-AUC k-mers: 0.3480
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.73      0.83      0.77       428
           1       0.35      0.23      0.27       171

    accuracy                           0.66       599
   macro avg       0.54      0.53      0.52       599
weighted avg       0.62      0.66      0.63       599

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.73      0.83      0.77       428
           1       0.33      0.21      0.26       171

    accuracy                           0.65       599
   macro avg       0.53      0.52      0.52       599
weighted avg       0.61      0.65      0.63       599



In [3]:
#PRUEBA LSTM BIDIRECCIONAL CON BASE REDUCIDA

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.metrics import AUC

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_50_OR_2400_reducida_extendida.csv")

etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)

X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

dicc = {"a": 1, "c": 2, "g": 3, "t": 4}

def codificador(secuencia):
    return np.array([dicc.get(nuc, 0) for nuc in secuencia])

X_cod = np.array([codificador(secuencia) for secuencia in X_seq])

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 2026)
id_train_val, id_test = next(gss1.split(X_cod, y, groups = grupos_rsid))

X_tv = X_cod[id_train_val]
y_tv = y[id_train_val]
grupos_tv = grupos_rsid[id_train_val]

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
id_train, id_val = next(gss2.split(X_tv, y_tv, groups = grupos_tv))

X_train, y_train = X_tv[id_train], y_tv[id_train]
X_val, y_val = X_tv[id_val], y_tv[id_val]
X_test, y_test = X_cod[id_test], y[id_test]

pesos_clases = compute_class_weight(class_weight = "balanced", classes = np.unique(y_train), y = y_train)

dicc_pesos = {0: pesos_clases[0], 1: pesos_clases[1]}

longitud_sec = len(base.iloc[0]["Secuencia"])
tam_vocabulario = 5
dim_embedding = 16

model = Sequential([

    Embedding(input_dim = tam_vocabulario,
              output_dim = dim_embedding,
              input_length = longitud_sec),

    Bidirectional(LSTM(units = 32, dropout = 0.3, recurrent_dropout = 0.3)),

    Dropout(0.3),

    Dense(units = 1, activation = "sigmoid")
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss = "binary_crossentropy",
    metrics = ["accuracy", AUC(name = "auc_roc"), AUC(name = "auc_pr", curve = "PR")]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = "val_auc_pr",
    mode = "max",
    patience = 10,
    restore_best_weights = True
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = 50, 
    batch_size = 64, 
    class_weight = dicc_pesos,
    callbacks = [early_stop],
    verbose = 1
)

print("RESULTADOS FINALES")

y_pred_proba = model.predict(X_test).flatten()
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

print(f"ROC-AUC Test: {roc_auc_score(y_test, y_pred_proba): .4f}")
print(f"PR-AUC Test: {average_precision_score(y_test, y_pred_proba): .4f}\n")

print(classification_report(y_test, y_pred_bin))

2026-06-04 20:57:32.345232: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-04 20:57:32.382092: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-04 20:57:32.963397: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-06-04 20:57:33.464880: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-06-04 20:57:33.464905: I tensorflow/compiler/xla/stream_executor/cuda/

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 101, 16)           80        
                                                                 
 bidirectional (Bidirection  (None, 64)                12544     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 12689 (49.57 KB)
Trainable params: 12689 (49.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/50
23/23 [==============================] - 5s 84ms/step - loss: 0.6924 - accu

In [4]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON RandomForestClassifier y VENTANA 500

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_500_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()


model_oh = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)
model_kmers = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5716
ROC-AUC k-mers: 0.6815
PR-AUC One-Hot: 0.2565
PR-AUC k-mers: 0.3451
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.61      0.78      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.78      1.00      0.87       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.60      0.78      0.68       770



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [5]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier Y VENTANA 500

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_500_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()

scaler = StandardScaler()
X_train_kmers = scaler.fit_transform(X_train_kmers)
X_test_kmers = scaler.transform(X_test_kmers)



model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5657
ROC-AUC k-mers: 0.6530
PR-AUC One-Hot: 0.2737
PR-AUC k-mers: 0.3274
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      0.98      0.87       599
           1       0.22      0.02      0.04       171

    accuracy                           0.76       770
   macro avg       0.50      0.50      0.45       770
weighted avg       0.65      0.76      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.79      0.96      0.87       599
           1       0.42      0.10      0.16       171

    accuracy                           0.77       770
   macro avg       0.61      0.53      0.51       770
weighted avg       0.71      0.77      0.71       770



In [6]:
#PRUEBA LSTM BIDIRECCIONAL CON VENTANA 500

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.metrics import AUC

base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_500_OR_3109_con_rsid_extendida.csv")

etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)

X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

dicc = {"a": 1, "c": 2, "g": 3, "t": 4}

def codificador(secuencia):
    return np.array([dicc.get(nuc, 0) for nuc in secuencia])

X_cod = np.array([codificador(secuencia) for secuencia in X_seq])

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 2026)
id_train_val, id_test = next(gss1.split(X_cod, y, groups = grupos_rsid))

X_tv = X_cod[id_train_val]
y_tv = y[id_train_val]
grupos_tv = grupos_rsid[id_train_val]

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
id_train, id_val = next(gss2.split(X_tv, y_tv, groups = grupos_tv))

X_train, y_train = X_tv[id_train], y_tv[id_train]
X_val, y_val = X_tv[id_val], y_tv[id_val]
X_test, y_test = X_cod[id_test], y[id_test]

pesos_clases = compute_class_weight(class_weight = "balanced", classes = np.unique(y_train), y = y_train)

dicc_pesos = {0: pesos_clases[0], 1: pesos_clases[1]}

longitud_sec = len(base.iloc[0]["Secuencia"])
tam_vocabulario = 5
dim_embedding = 16

model = Sequential([

    Embedding(input_dim = tam_vocabulario,
              output_dim = dim_embedding,
              input_length = longitud_sec),

    Bidirectional(LSTM(units = 32, dropout = 0.3, recurrent_dropout = 0.3)),

    Dropout(0.3),

    Dense(units = 1, activation = "sigmoid")
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
    loss = "binary_crossentropy",
    metrics = ["accuracy", AUC(name = "auc_roc"), AUC(name = "auc_pr", curve = "PR")]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor = "val_auc_pr",
    mode = "max",
    patience = 10,
    restore_best_weights = True
)

history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = 50, 
    batch_size = 64, 
    class_weight = dicc_pesos,
    callbacks = [early_stop],
    verbose = 1
)

print("RESULTADOS FINALES")

y_pred_proba = model.predict(X_test).flatten()
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

print(f"ROC-AUC Test: {roc_auc_score(y_test, y_pred_proba): .4f}")
print(f"PR-AUC Test: {average_precision_score(y_test, y_pred_proba): .4f}\n")

print(classification_report(y_test, y_pred_bin))

2026-06-05 19:08:51.190163: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-05 19:08:51.225746: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-05 19:08:51.832400: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-06-05 19:08:52.621912: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:268] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-06-05 19:08:52.621936: I tensorflow/compiler/xla/stream_executor/cuda/

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 1001, 16)          80        
                                                                 
 bidirectional (Bidirection  (None, 64)                12544     
 al)                                                             
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 12689 (49.57 KB)
Trainable params: 12689 (49.57 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/50
30/30 [==============================] - 22s 646ms/step - loss: 0.6935 - ac